# 02 — Data Understanding & Cleaning

Covers project spec sections 2 (Data Understanding) and 3 (Data Cleaning).

Every decision below is explained. No rows are deleted blindly.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")
from team_name_mapping import standardize_team_name

pd.set_option("display.max_columns", 20)

matches = pd.read_csv("../data/raw/results.csv")
ranking = pd.read_csv("../data/raw/fifa_ranking.csv")
print("matches:", matches.shape)
print("ranking:", ranking.shape)


matches: (49520, 9)
ranking: (62424, 9)


## 2. Data Understanding

In [2]:
matches.info()


<class 'pandas.DataFrame'>
RangeIndex: 49520 entries, 0 to 49519
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        49520 non-null  str  
 1   home_team   49520 non-null  str  
 2   away_team   49520 non-null  str  
 3   home_score  49520 non-null  int64
 4   away_score  49520 non-null  int64
 5   tournament  49520 non-null  str  
 6   city        49520 non-null  str  
 7   country     49520 non-null  str  
 8   neutral     49520 non-null  bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 3.1 MB


In [3]:
matches.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
date,49520,16491,2012-02-29,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
home_team,49520,328,Brazil,618,NaN,NaN,NaN,NaN,NaN,NaN,NaN
away_team,49520,322,Uruguay,585,NaN,NaN,NaN,NaN,NaN,NaN,NaN
home_score,49520.0,NaN,NaN,NaN,1.757209,1.773726,0.0,1.0,1.0,2.0,31.0
away_score,49520.0,NaN,NaN,NaN,1.182411,1.402076,0.0,0.0,1.0,2.0,21.0
tournament,49520,202,Friendly,18384,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,49520,2092,Kuala Lumpur,745,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,49520,269,United States,1585,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neutral,49520,2,False,36364,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print("Missing values (matches):")
print(matches.isnull().sum())
print()
print("Duplicate rows (matches):", matches.duplicated().sum())
print("Unique home teams:", matches.home_team.nunique())
print("Unique away teams:", matches.away_team.nunique())
print("Unique tournaments:", matches.tournament.nunique())
print("Date range:", matches.date.min(), "to", matches.date.max())


Missing values (matches):
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

Duplicate rows (matches): 0
Unique home teams: 328
Unique away teams: 322
Unique tournaments: 202
Date range: 1872-11-30 to 2026-07-19


In [5]:
ranking.info()


<class 'pandas.DataFrame'>
RangeIndex: 62424 entries, 0 to 62423
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               62424 non-null  int64
 1   rank             62424 non-null  int64
 2   country_full     62424 non-null  str  
 3   country_abrv     62424 non-null  str  
 4   total_points     62424 non-null  int64
 5   previous_points  62424 non-null  int64
 6   rank_change      62424 non-null  int64
 7   confederation    62424 non-null  str  
 8   rank_date        62424 non-null  str  
dtypes: int64(5), str(4)
memory usage: 4.3 MB


In [6]:
print("Missing values (ranking):")
print(ranking.isnull().sum())
print()
print("Duplicate rows (ranking):", ranking.duplicated().sum())
print("Unique countries:", ranking.country_full.nunique())
print("Rank snapshot dates:", ranking.rank_date.nunique())
print("Date range:", ranking.rank_date.min(), "to", ranking.rank_date.max())


Missing values (ranking):
id                 0
rank               0
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64



Duplicate rows (ranking): 0
Unique countries: 216
Rank snapshot dates: 309
Date range: 1992-12-31 to 2020-12-10


### Suspicious values check

Before cleaning, look for impossible scores, invalid ranks, and inconsistent naming.

In [7]:
print("Max home_score:", matches.home_score.max(), "| Max away_score:", matches.away_score.max())
print("Any negative scores?", (matches[["home_score","away_score"]] < 0).any().any())
print("Rows with score > 20 (sanity check, not necessarily wrong):")
display_cols = ["date","home_team","away_team","home_score","away_score","tournament"]
matches[(matches.home_score > 20) | (matches.away_score > 20)][display_cols]


Max home_score: 31 | Max away_score: 21
Any negative scores? False
Rows with score > 20 (sanity check, not necessarily wrong):


,date,home_team,away_team,home_score,away_score,tournament
6580,1966-04-03,Libya,Oman,21,0,Arab Cup
8551,1971-09-13,Tahiti,Cook Islands,30,0,South Pacific Games
11916,1979-08-30,Fiji,Kiribati,24,0,South Pacific Games
25422,2001-04-09,Australia,Tonga,22,0,FIFA World Cup qualification
25425,2001-04-11,Australia,American Samoa,31,0,FIFA World Cup qualification
29045,2005-03-11,Guam,North Korea,0,21,EAFF Championship
30521,2006-11-24,Sápmi,Monaco,21,1,Viva World Cup
37062,2013-06-23,Quebec,Tibet,21,0,"International Tournament of Peoples, Cultures ..."
37064,2013-06-24,Provence,Tibet,22,0,"International Tournament of Peoples, Cultures ..."


In [8]:
print("Rank value range:", ranking["rank"].min(), "-", ranking["rank"].max())
print("Any rank <= 0?", (ranking["rank"] <= 0).any())
print("Any negative total_points?", (ranking.total_points < 0).any())


Rank value range: 1 - 211
Any rank <= 0? False
Any negative total_points? False


**Finding:** the handful of very lopsided scores (e.g. 20+ goals) are all real historical blowouts
in early-era qualifiers/friendlies between mismatched opponents (a well-documented pattern in this
dataset — e.g. Australia 31-0 American Samoa, 2001). These are not data errors, so they are **kept**,
not deleted. Ranks and points are all within valid ranges — no invalid values found.

## 3. Data Cleaning

### 3.1 Duplicates
No exact duplicate rows were found in either dataset (checked above), so nothing to drop here.

In [9]:
assert matches.duplicated().sum() == 0
assert ranking.duplicated().sum() == 0
print("Confirmed: 0 exact duplicates in both datasets.")


Confirmed: 0 exact duplicates in both datasets.


### 3.2 Standardize team / country names

We diffed the two datasets' team-name sets directly (not guessed) and found 140 names present in
`matches` but absent from `ranking`. Most are genuinely non-FIFA entities (regional teams,
historic/defunct states, micronations) that correctly have no FIFA ranking and should NOT be
force-matched. A confirmed subset are real naming-convention differences (e.g. "Ivory Coast" vs
"Côte d'Ivoire"), which we fix with an explicit mapping — see `src/team_name_mapping.py`.

In [10]:
matches["home_team_std"] = matches["home_team"].apply(standardize_team_name)
matches["away_team_std"] = matches["away_team"].apply(standardize_team_name)

mset = set(matches.home_team_std) | set(matches.away_team_std)
rset = set(ranking.country_full)
still_unmatched = mset - rset
print(f"Unmatched team names before mapping: 140")
print(f"Unmatched team names after mapping: {len(still_unmatched)}")
print("(remaining ones are genuinely non-FIFA entities, e.g. regional/historic teams -- expected)")


Unmatched team names before mapping: 140
Unmatched team names after mapping: 127
(remaining ones are genuinely non-FIFA entities, e.g. regional/historic teams -- expected)


### 3.3 Convert dates to datetime

In [11]:
matches["date"] = pd.to_datetime(matches["date"])
ranking["rank_date"] = pd.to_datetime(ranking["rank_date"])
print(matches["date"].dtype, ranking["rank_date"].dtype)


datetime64[us]

 datetime64[us]


### 3.4 Ensure chronological ordering

Critical for this project: everything downstream (form, Elo, chronological train/test split)
depends on strict time ordering.

In [12]:
matches = matches.sort_values("date").reset_index(drop=True)
ranking = ranking.sort_values(["rank_date", "rank"]).reset_index(drop=True)
print("matches sorted:", matches["date"].is_monotonic_increasing)
print("ranking sorted:", ranking["rank_date"].is_monotonic_increasing)


matches sorted: True
ranking sorted: True


### 3.5 Missing values strategy

Both source datasets have **0% missing values** in every column (verified above), so there is no
imputation to do at this stage. Missingness will only appear later, once we engineer features
like "last-5-match form" or "FIFA rank at match time" for teams/dates where no prior data exists
(e.g. a team's very first international match, or matches before FIFA rankings started in Dec 1992).

For those cases, our policy (documented for section 5 / feature engineering, not applied here):
- **Rolling form features (last-5/last-10):** left as NaN if a team has fewer than the required
  prior matches. We will NOT forward-fill or impute these — a genuinely new national team having
  no "form" is real information, not missing data to be guessed at.
- **FIFA ranking pre-Dec-1992:** left as NaN. FIFA rankings did not exist before then, so there is
  no legitimate value to impute. Models will either restrict training to the post-1992 era for
  ranking-dependent features, or use a separate "has_ranking" flag.
- We explicitly reject mean/median imputation for these two cases since it would inject
  fabricated information exactly where the project's DATA REQUIREMENT prohibits it.

### 3.6 Remove irrelevant columns

`city` and `country` (venue location) are kept for now — they're needed to compute the
`neutral_venue` feature correctly downstream — but are not core modeling columns yet.
Nothing is dropped at the cleaning stage; column selection happens in feature engineering.

### 3.7 Outlier check on ranking points

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ranking["total_points"], bins=50, color="#2b6cb0")
axes[0].set_title("Distribution of FIFA ranking points")
axes[0].set_xlabel("total_points")
axes[0].set_ylabel("count")

goal_diff = matches["home_score"] - matches["away_score"]
axes[1].hist(goal_diff, bins=range(-15, 16), color="#2f855a")
axes[1].set_title("Distribution of goal difference (home - away)")
axes[1].set_xlabel("goal difference")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.savefig("../visualizations/02_cleaning_sanity_checks.png", dpi=110)
plt.show()
print("Saved to ../visualizations/02_cleaning_sanity_checks.png")


Saved to ../visualizations/02_cleaning_sanity_checks.png


**Interpretation:** ranking points are roughly bell-shaped with a long right tail (a handful of
elite teams), which is expected and not an error. Goal difference is centered slightly above 0,
consistent with the well-documented home-advantage effect in football — this will be an important
feature later, not a data quality issue.

## Save processed data

In [14]:
matches.to_csv("../data/processed/matches_clean.csv", index=False)
ranking.to_csv("../data/processed/fifa_ranking_clean.csv", index=False)
print("Saved:")
print(" - data/processed/matches_clean.csv", matches.shape)
print(" - data/processed/fifa_ranking_clean.csv", ranking.shape)


Saved:
 - data/processed/matches_clean.csv (49520, 11)
 - data/processed/fifa_ranking_clean.csv (62424, 9)
